# 🧹 Notebook 02: Limpeza, Tipagem e Feature Engineering
### Projeto: Análise e Diagnóstico de Acidentes de Trânsito em Bauru/SP (2015–2026)

Neste notebook realizamos a higienização completa da base de Bauru:
1. Padronização de datas (`datetime`), extração de componentes temporais;
2. Limpeza de horários e turnos;
3. Conversão de coordenadas geográficas e validação de *bounding box*;
4. Criação de variáveis analíticas (*Feature Engineering*) para modais, gravidade e infraestrutura viária;
5. Exportação do dataset definitivo consolidado: `data/bauru_limpo.csv`.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# Ingestão das ocorrências brutas filtradas de Bauru
data_files = sorted(glob.glob('../data/sinistros_*.csv'))
dfs = []
for p in data_files:
    for chunk in pd.read_csv(p, sep=';', encoding='latin1', low_memory=False, chunksize=50000):
        cond = (chunk['cod_ibge'].astype(str) == '3506003') | (chunk['municipio'].astype(str).str.upper() == 'BAURU')
        cb = chunk[cond]
        if len(cb) > 0:
            dfs.append(cb)

df = pd.concat(dfs, ignore_index=True)
print(f"Base carregada: {len(df)} registros.")

## 1. Tratamento de Datas e Horas

In [ ]:
# Datas
df['data_sinistro_dt'] = pd.to_datetime(df['data_sinistro'], format='%d/%m/%Y', errors='coerce')
df['ano'] = pd.to_numeric(df['ano_sinistro'], errors='coerce').fillna(df['data_sinistro_dt'].dt.year).astype(int)
df['mes'] = pd.to_numeric(df['mes_sinistro'], errors='coerce').fillna(df['data_sinistro_dt'].dt.month).astype(int)

# Horas
def parse_hora(h):
    if pd.isna(h):
        return np.nan
    s = str(h).strip()
    if ':' in s:
        try:
            return int(s.split(':')[0])
        except:
            return np.nan
    return np.nan

df['hora_num'] = df['hora_sinistro'].apply(parse_hora)

# Turnos padronizados
def classificar_turno(h):
    if pd.isna(h):
        return 'NAO INFORMADO'
    if 0 <= h < 6:
        return 'MADRUGADA'
    elif 6 <= h < 12:
        return 'MANHA'
    elif 12 <= h < 18:
        return 'TARDE'
    else:
        return 'NOITE'

df['turno_ajustado'] = df['hora_num'].apply(classificar_turno)
print("Distribuição por turno:", df['turno_ajustado'].value_counts().to_dict())

## 2. Tratamento de Coordenadas Geográficas (Latitude e Longitude)

In [ ]:
lat_str = df['latitude'].astype(str).str.replace(',', '.').str.strip()
lon_str = df['longitude'].astype(str).str.replace(',', '.').str.strip()
df['lat'] = pd.to_numeric(lat_str, errors='coerce')
df['lon'] = pd.to_numeric(lon_str, errors='coerce')

# Bounding box territorial de Bauru (-22.6 a -22.0 lat, -49.4 a -48.8 lon)
df['geo_valida'] = (df['lat'].between(-22.6, -22.0)) & (df['lon'].between(-49.4, -48.8))
print(f"Pontos georreferenciados válidos: {df['geo_valida'].sum()} ({df['geo_valida'].sum()/len(df)*100:.1f}%)")

## 3. Feature Engineering: Modais, Severidade e Vias

In [ ]:
# Conversão segura de colunas numéricas
cols_qtd = [
    'qtd_gravidade_fatal', 'qtd_gravidade_grave', 'qtd_gravidade_leve', 'qtd_gravidade_ileso',
    'qtd_pedestre', 'qtd_bicicleta', 'qtd_motocicleta', 'qtd_automovel', 'qtd_onibus', 'qtd_caminhao'
]
for col in cols_qtd:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df['obitos'] = df['qtd_gravidade_fatal']
df['feridos_graves'] = df['qtd_gravidade_grave']
df['feridos_leves'] = df['qtd_gravidade_leve']

# Flags binárias de modais
df['envolve_moto'] = (df['qtd_motocicleta'] > 0).astype(int)
df['envolve_auto'] = (df['qtd_automovel'] > 0).astype(int)
df['envolve_pedestre'] = (df['qtd_pedestre'] > 0).astype(int)
df['envolve_bicicleta'] = (df['qtd_bicicleta'] > 0).astype(int)
df['envolve_pesado'] = ((df['qtd_caminhao'] > 0) | (df['qtd_onibus'] > 0)).astype(int)

# Severidade binária (Grave ou Fatal)
df['severidade_grave_ou_fatal'] = ((df['obitos'] > 0) | (df['feridos_graves'] > 0)).astype(int)

# Classificação categórica de severidade
def classificar_severidade(r):
    if r['obitos'] > 0:
        return 'FATAL'
    elif r['feridos_graves'] > 0:
        return 'GRAVE'
    elif r['feridos_leves'] > 0:
        return 'LEVE'
    return 'ILESO / NAO INFORMADO'

df['severidade_categoria'] = df.apply(classificar_severidade, axis=1)

# Dia da semana e final de semana
mapa_dias = {
    'SEGUNDA-FEIRA': 'Segunda-feira', 'TERCA-FEIRA': 'Terça-feira', 'TERÇA-FEIRA': 'Terça-feira',
    'QUARTA-FEIRA': 'Quarta-feira', 'QUINTA-FEIRA': 'Quinta-feira', 'SEXTA-FEIRA': 'Sexta-feira',
    'SABADO': 'Sábado', 'SÁBADO': 'Sábado', 'DOMINGO': 'Domingo'
}
df['dia_da_semana_limpo'] = df['dia_da_semana'].astype(str).str.upper().str.strip().map(lambda x: mapa_dias.get(x, x.capitalize()))
df['is_fim_de_semana'] = df['dia_da_semana_limpo'].isin(['Sábado', 'Domingo']).astype(int)

# Classificação viária (Vias Urbanas vs Rodovias/Estradas)
def classificar_via(r):
    via = str(r['tipo_via']).upper()
    admin = str(r['administracao']).upper()
    if 'RODOVIA' in via or 'ESTRADA' in via or 'DER' in admin or 'CONCESSION' in admin:
        return 'RODOVIA / ESTRADA'
    return 'VIA URBANA'

df['tipo_via_simplificado'] = df.apply(classificar_via, axis=1)

# Tipo de sinistro simplificado
def classificar_tipo_sinistro(tp):
    s = str(tp).upper()
    if 'CHOQUE' in s:
        return 'CHOQUE'
    elif 'ATROPEL' in s:
        return 'ATROPELAMENTO'
    elif 'COLIS' in s:
        return 'COLISAO'
    elif 'CAPOTAMENTO' in s or 'TOMBAMENTO' in s:
        return 'CAPOTAMENTO/TOMBAMENTO'
    elif 'OUTROS' in s:
        return 'OUTROS'
    return 'OUTROS / NAO INFORMADO'

df['tipo_sinistro_simplificado'] = df['tp_sinistro_primario'].apply(classificar_tipo_sinistro)

## 4. Exportação do Dataset Higienizado

In [ ]:
caminho_saida = '../data/bauru_limpo.csv'
df.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8')
print(f"Dataset consolidado e limpo salvo com sucesso em {caminho_saida} ({len(df)} registros).")